In [ ]:
# 1. 환경 설정
# YOLOv8 설치
!pip install ultralytics

# 라이브러리 임포트
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow


In [ ]:
# 2. 사전 학습 모델로 예측
# 모델 로드 (사전 학습된 경량 세그멘테이션 모델)
model = YOLO('yolov8n-seg.pt')

# 이미지 예측
results = model.predict(source='https://ultralytics.com/images/bus.jpg', conf=0.3)

# 결과 시각화
for r in results:
    im_array = r.plot()  # 마스크와 바운딩박스가 그려진 이미지
    cv2_imshow(im_array)


In [ ]:
# 3. 커스텀 데이터 학습
# data.yaml 파일은 Google Drive나 Colab 환경에 준비되어 있어야 함
# 예시: data.yaml 안에 train/val 경로와 클래스 이름 정의
model = YOLO('yolov8n-seg.pt')

model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640
)


In [ ]:
# 4. 실시간 웹캠 세그멘테이션
# Colab에서 웹캠 사용
# 실행 후 '권한 허용'을 눌러야 함
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);

        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});

        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Resize output to fit the video element.
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        // Wait for Capture to be clicked.
        await new Promise((resolve) => capture.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
      takePhoto({{quality}});
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

# 사진 촬영
photo_path = take_photo()

# 예측
results = model.predict(source=photo_path, conf=0.3)
for r in results:
    im_array = r.plot()
    cv2_imshow(im_array)


In [ ]:
# 5. 결과 저장
# 예측 결과 저장
results[0].save(filename='segmentation_result.jpg')
print("결과 저장 완료!")


💡 팁
Colab에서 웹캠 실시간 스트리밍은 제약이 있으니, 로컬 환경에서 source=0으로 실행하면 더 부드럽게 동작합니다.
커스텀 데이터 학습 시 data.yaml 경로와 이미지 폴더 구조를 정확히 맞춰야 합니다.


좋아요 👍 아래는 YOLOv8-Seg 커스텀 데이터 학습을 위한 data.yaml 예시와 폴더 구조 설명입니다. 이대로 구성하면 Colab이나 로컬에서 바로 학습을 시작할 수 있습니다.

📂 폴더 구조 예시
yolov8_seg_dataset/          # 데이터셋 루트
│
├── data.yaml                # 데이터셋 설정 파일
│
├── images/
│   ├── train/
│   │   ├── img001.jpg
│   │   └── img002.jpg
│   └── val/
│       ├── img101.jpg
│       └── img102.jpg
│
└── labels/
    ├── train/
    │   ├── img001.txt
    │   └── img002.txt
    └── val/
        ├── img101.txt
        └── img102.txt

📝 data.yaml 예시
# YOLOv8-Segmentation custom dataset configuration

# 데이터셋 루트 경로
path: yolov8_seg_dataset

# 학습/검증 이미지 경로
train: images/train
val: images/val

# 클래스 개수
nc: 2

# 클래스 이름
names: ["cat", "dog"]


📌 라벨 파일(.txt) 형식
YOLOv8-Seg 라벨은 객체 클래스 + 마스크 좌표 형식입니다.

<class_id> x1 y1 x2 y2 x3 y3 ... xn yn

class_id : 0부터 시작하는 클래스 번호
(x, y) : 이미지 크기에 대해 0~1로 정규화된 마스크 꼭짓점 좌표
한 객체당 한 줄, 여러 객체가 있으면 여러 줄

💡 팁
images/와 labels/의 파일명은 반드시 동일해야 합니다. (img001.jpg ↔ img001.txt)
마스크 라벨링은 Labelme 또는 Roboflow 같은 툴을 쓰면 편리합니다.


yolo task=segment mode=train model=yolov8n-seg.pt data=yolov8_seg_dataset/data.yaml epochs=50 imgsz=640
